In [ ]:
import os
import re
import gc
import json
import time
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# -------------------- Настройки --------------------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [3]:
LOCAL_FILES = {
    'train': 'topiocqa_train.json',
    'dev': 'topiocqa_dev.json'
}
LOCAL_SAVE_PATH = "./topiocqa_ru"
CACHE_FILE = "translation_cache_topiocqa.jsonl"

# Поля, которые нужно перевести
FIELDS_TO_TRANSLATE = ['Context', 'Question', 'Answer', 'Topic', 'Topic_section', 'Rationale']

In [4]:
# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()
print(f"Записей в кэше: {len(translation_cache)}")

Записей в кэше: 0


In [8]:
# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # Попытка перевода целиком
    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                break
            if any(c in error_str for c in ['502','503','504']):
                time.sleep(delay * (attempt+1))
            elif '429' in error_str:
                time.sleep(delay*4 + 10)
            else:
                time.sleep(delay)

    # Разбиение на предложения
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1:
        return _translate_parts(sentences, retries, delay, text)

    # Длинный текст — принудительное разбиение
    if len(text) > 4000:
        return _force_split(text, 3500, retries, delay)

    # Короткий текст, который не перевёлся — возвращаем оригинал
    return text, True

def _translate_parts(parts, retries, delay, original):
    translated = []
    for part in parts:
        if not part.strip():
            translated.append(part)
            continue
        t, _ = translate_text_robust(part, retries, delay)
        translated.append(t if t else part)
    full = ' '.join(translated)
    translation_cache[original] = full
    append_cache(original, full)
    return full, True

def _force_split(text, chunk_size, retries, delay):
    words = text.split()
    chunks = []
    cur_chunk = []
    cur_len = 0
    for word in words:
        cur_chunk.append(word)
        cur_len += len(word) + 1
        if cur_len >= chunk_size:
            chunks.append(' '.join(cur_chunk))
            cur_chunk = []
            cur_len = 0
    if cur_chunk:
        chunks.append(' '.join(cur_chunk))

    if len(chunks) <= 1:
        return text, True

    logging.info(f"Force splitting into {len(chunks)} chunks")
    translated = []
    for chunk in chunks:
        t, _ = translate_text_robust(chunk, retries, delay)
        translated.append(t if t else chunk)
    full = ' '.join(translated)
    translation_cache[text] = full
    append_cache(text, full)
    return full, True

# -------------------- Перевод одного примера --------------------
def translate_example(example):
    result = {}
    all_ok = True

    for key, value in example.items():
        if key == 'Additional_answers':
            # Всегда записываем пустой список
            result[key] = []
            continue

        if key in FIELDS_TO_TRANSLATE:
            if isinstance(value, str):
                trans, ok = translate_text_robust(value)
                if not ok:
                    all_ok = False
                result[key] = trans if ok else value
            elif isinstance(value, list):
                # Context — список строк
                trans_list = []
                for item in value:
                    if isinstance(item, str):
                        t, ok = translate_text_robust(item)
                        if not ok:
                            all_ok = False
                        trans_list.append(t if ok else item)
                    else:
                        trans_list.append(item)
                result[key] = trans_list
            else:
                result[key] = value
        else:
            # Непереводимые поля: Conversation_no, Turn_no, is_nq
            result[key] = value

    return result, all_ok

# -------------------- Обработка одного файла --------------------
def process_file(split_name, input_file):
    progress_file = f"translated_topiocqa_{split_name}.jsonl"

    # Загружаем исходный JSON
    with open(input_file, 'r', encoding='utf-8') as f:
        all_data = json.load(f)

    total = len(all_data)
    logging.info(f"[{split_name}] Всего примеров: {total}")

    # Проверяем прогресс
    start_idx = 0
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            start_idx = sum(1 for _ in f)
        logging.info(f"[{split_name}] Уже обработано: {start_idx}")

    if start_idx >= total:
        logging.info(f"[{split_name}] Все записи уже обработаны.")
        return

    # Обрабатываем и дописываем
    with open(progress_file, "a", encoding="utf-8") as out:
        pbar = tqdm(
            enumerate(all_data[start_idx:], start=start_idx),
            initial=start_idx,
            total=total,
            desc=f"Translating {split_name}"
        )
        for i, example in pbar:
            translated, ok = translate_example(example)
            record = {
                '_index': i,
                '_failed': not ok,
                **translated
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

            # Сборщик мусора каждые 200 записей
            if (i - start_idx + 1) % 200 == 0:
                gc.collect()

    logging.info(f"[{split_name}] Готово. Прогресс сохранён в {progress_file}")

In [ ]:
# -------------------- Запуск --------------------
for split, filename in LOCAL_FILES.items():
    if not os.path.exists(filename):
        logging.warning(f"Файл {filename} не найден, пропускаем {split}")
        continue
    process_file(split, filename)

# Финальная очистка памяти
gc.collect()
print("\nПеревод завершён!")

In [ ]:
import os, json
from datasets import Dataset, DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")

REPO_ID = "DeepPavlov/topiocqa_ru"

final_splits = {}
for split in ['train', 'dev']:
    progress_file = f"translated_topiocqa_{split}.jsonl"
    if not os.path.exists(progress_file):
        print(f"Файл {progress_file} не найден, пропускаем")
        continue
    
    clean_records = []
    with open(progress_file, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            rec.pop('_index', None)
            rec.pop('_failed', None)
            # Принудительно удаляем Additional_answers, если оно есть
            rec.pop('Additional_answers', None)
            clean_records.append(rec)
    
    final_splits[split] = Dataset.from_list(clean_records)
    print(f"{split}: {len(clean_records)} примеров")

dataset = DatasetDict(final_splits)

# Сохраняем локально
dataset.save_to_disk("./topiocqa_ru")
print("\nДатасет сохранён локально")

# Загружаем на HF
dataset.push_to_hub(
    REPO_ID,
    private=False,
    commit_message="Russian translation – all fields translated, Additional_answers removed"
)

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")